# 02 — Data preparation

## Purpose

In Notebook 01, we selected the **5-minute SOLETE data** for short-horizon solar-power forecasting. Before building models, we need to prepare a reliable set of historical observations.

This notebook checks and corrects three known measurement issues:

- pressure values with an extra 1000 mbar at midnight;
- invalid humidity values in a series recorded as fractions;
- wind directions that need circular encoding.

We keep the original measurements unchanged and add cleaned columns. The forecasting target is **solar PV power**, `P_Solar[kW]`.

This notebook prepares observational variables that will be used at or before the forecast issue time.

## 1. Load the 5-minute measurements

We use UTC timestamps throughout the study. A complete, ordered timeline is essential when we later create historical lags and future forecasting targets.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

# Find the project directory when working in VS Code.
PROJECT_ROOT = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "solete_dataset" / "SOLETE_Pombo_5min.h5").is_file()
        and (p / "src" / "preprocessing.py").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project containing solete_dataset/ and src/.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete
from src.preprocessing import (
    build_cleaned_solete_features,
    clean_humidity_fraction,
    clean_pressure_mbar,
    clean_wind_direction_deg,
    wind_direction_cyclic_features,
)

path = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_5min.h5"
raw = load_solete(path).sort_index()

assert isinstance(raw.index, pd.DatetimeIndex)
assert str(raw.index.tz) == "UTC"
assert raw.index.is_unique and raw.index.is_monotonic_increasing
assert (raw.index.to_series().diff().dropna() == pd.Timedelta(minutes=5)).all()
assert "P_Solar[kW]" in raw.columns

print("Dataset shape:", raw.shape)
print("Time period:", raw.index.min(), "to", raw.index.max())
print("Missing source values:", int(raw.isna().sum().sum()))

Dataset shape: (131617, 11)
Time period: 2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Missing source values: 0


## 2. Identify measurements that need correction

The data exploration found three specific issues. We count them here before making any changes.

- **Pressure:** values above 1100 mbar belong to a known SOLETE midnight artifact.
- **Humidity:** values above 1 are invalid for this fraction-scaled series.
- **Wind direction:** angles outside 0–360° should be wrapped, not removed.

The pressure and humidity decisions are **dataset-specific**; these thresholds must not be applied automatically to other datasets.

In [2]:
pressure = raw["Pressure[mbar]"]
humidity = raw["HUMIDITY[%]"]
wind_direction = raw["WIND_DIR[deg]"]

pressure_issue = pressure > 1100
humidity_issue = humidity > 1
wind_issue = (wind_direction < 0) | (wind_direction >= 360)
midnight = (raw.index.hour == 0) & (raw.index.minute == 0)

issues = pd.DataFrame(
    {
        "affected_rows": [int(pressure_issue.sum()), int(humidity_issue.sum()), int(wind_issue.sum())],
        "midnight_only": [
            bool((~pressure_issue | midnight).all()),
            bool((~humidity_issue | midnight).all()),
            "Not applicable",
        ],
    },
    index=["Pressure above 1100 mbar", "Humidity above 1", "Wind direction outside [0, 360)"],
)
display(issues)

assert pressure_issue.any() and humidity_issue.any()
assert bool(issues.loc["Pressure above 1100 mbar", "midnight_only"])
assert bool(issues.loc["Humidity above 1", "midnight_only"])

,affected_rows,midnight_only
Pressure above 1100 mbar,456,True
Humidity above 1,164,True
"Wind direction outside [0, 360)",103,Not applicable


## 3. Correct the pressure artifact

A group of pressure readings has approximately **1000 mbar added** at midnight. Subtracting 1000 from these readings restores their usual scale while keeping the measured variation.

We check the values against the preceding observation to illustrate the problem. The correction itself uses only the current reading, not future measurements.

In [3]:
pressure_clean = clean_pressure_mbar(pressure)
previous_pressure = pressure.shift(1)

pressure_check = pd.Series(
    {
        "flagged_rows": int(pressure_issue.sum()),
        "median_raw_jump_from_previous_mbar": float((pressure - previous_pressure).loc[pressure_issue].median()),
        "median_clean_jump_from_previous_mbar": float((pressure_clean - previous_pressure).loc[pressure_issue].median()),
        "cleaned_min_mbar": float(pressure_clean.min()),
        "cleaned_max_mbar": float(pressure_clean.max()),
    },
    name="Pressure correction",
)
display(pressure_check.to_frame("value"))

# Every unflagged reading must be identical to the source value.
assert pressure_clean.loc[~pressure_issue].equals(pressure.loc[~pressure_issue])
np.testing.assert_allclose(
    pressure_clean.loc[pressure_issue].to_numpy(),
    pressure.loc[pressure_issue].to_numpy() - 1000.0,
)
assert pressure_clean.notna().all()

,value
flagged_rows,456.000000
median_raw_jump_from_previous_mbar,1000.000000
median_clean_jump_from_previous_mbar,0.000000
cleaned_min_mbar,974.262999
cleaned_max_mbar,1042.319710


## 4. Replace invalid humidity using past information

Most humidity readings lie between 0 and 1, even though the source column name contains `%`. We therefore treat the series as a fraction.

For readings above 1, we use the **most recent valid value**. This is called *forward filling*. It is appropriate for a forecasting input because it does not look at future readings.

In [4]:
humidity_clean = clean_humidity_fraction(humidity)

# Independently reproduce the causal replacement rule.
expected_humidity = humidity.mask(humidity_issue).ffill()
pd.testing.assert_series_equal(humidity_clean, expected_humidity)

assert humidity_clean.notna().all()
assert humidity_clean.between(0, 1).all()
assert humidity_clean.loc[~humidity_issue].equals(humidity.loc[~humidity_issue])

humidity_check = pd.Series(
    {
        "replaced_rows": int(humidity_issue.sum()),
        "raw_min": float(humidity.min()),
        "raw_max": float(humidity.max()),
        "clean_min": float(humidity_clean.min()),
        "clean_max": float(humidity_clean.max()),
        "missing_after_cleaning": int(humidity_clean.isna().sum()),
    },
    name="Humidity cleaning",
)
display(humidity_check.to_frame("value"))

,value
replaced_rows,164.0
raw_min,0.0
raw_max,2.0
clean_min,0.0
clean_max,1.0
missing_after_cleaning,0.0


## 5. Represent wind direction correctly

Wind direction is circular: 359° and 1° are close together, even though their numbers are far apart.

We first wrap every angle into the range 0–360°, then represent it with sine and cosine. This avoids an artificial jump at north (0°/360°).

In [5]:
wind_wrapped = clean_wind_direction_deg(wind_direction)
wind_sin, wind_cos = wind_direction_cyclic_features(wind_wrapped)

assert wind_wrapped.ge(0).all() and wind_wrapped.lt(360).all()
assert wind_wrapped.loc[~wind_issue].equals(wind_direction.loc[~wind_issue])
np.testing.assert_allclose(
    wind_sin.to_numpy() ** 2 + wind_cos.to_numpy() ** 2,
    np.ones(len(raw)),
    rtol=0,
    atol=1e-12,
)

display(
    pd.DataFrame(
        {
            "raw_direction_deg": wind_direction.loc[wind_issue],
            "wrapped_direction_deg": wind_wrapped.loc[wind_issue],
            "sin": wind_sin.loc[wind_issue],
            "cos": wind_cos.loc[wind_issue],
        }
    ).head(5)
)

,raw_direction_deg,wrapped_direction_deg,sin,cos
Timestamp,,,,
2018-06-04 00:00:00+00:00,531.528428,171.528428,0.147319,-0.989089
2018-06-12 00:00:00+00:00,565.193980,205.193980,-0.425684,-0.904872
2018-06-13 00:00:00+00:00,489.896321,129.896321,0.767206,-0.641400
2018-06-14 00:00:00+00:00,470.468227,110.468227,0.936866,-0.349688
2018-06-18 00:00:00+00:00,457.521739,97.521739,0.991395,-0.130902


## 6. Assemble the cleaned observational data

We now apply the established preparation functions to the full 5-minute timeline. The original columns remain available alongside the cleaned columns.

The **solar PV target is not corrected, clipped, or smoothed**. Other observed weather and power variables also remain unchanged. This stage only cleans the selected pressure, humidity, and wind-direction predictors.

In [6]:
cleaned = build_cleaned_solete_features(raw)

# Confirm that the combined function uses the same corrections checked above.
pd.testing.assert_series_equal(cleaned["Pressure_clean[mbar]"], pressure_clean, check_names=False)
pd.testing.assert_series_equal(cleaned["Humidity_clean"], humidity_clean, check_names=False)
pd.testing.assert_series_equal(cleaned["Wind_dir_clean[deg]"], wind_wrapped, check_names=False)
np.testing.assert_allclose(cleaned["wind_direction_sin"], wind_sin)
np.testing.assert_allclose(cleaned["wind_direction_cos"], wind_cos)

# No source column is replaced or removed.
assert all(cleaned[column].equals(raw[column]) for column in raw.columns)
assert cleaned.index.equals(raw.index)

preview_columns = [
    "P_Solar[kW]", "Pressure[mbar]", "Pressure_clean[mbar]",
    "HUMIDITY[%]", "Humidity_clean", "WIND_DIR[deg]", "Wind_dir_clean[deg]",
]
display(cleaned.loc[pressure_issue | humidity_issue, preview_columns].head(5))
print("Prepared dataset shape:", cleaned.shape)

,P_Solar[kW],Pressure[mbar],Pressure_clean[mbar],HUMIDITY[%],Humidity_clean,WIND_DIR[deg],Wind_dir_clean[deg]
Timestamp,,,,,,,
2018-06-02 00:00:00+00:00,0.0,2013.659998,1013.659998,1.8,0.9,164.421405,164.421405
2018-06-03 00:00:00+00:00,0.0,2013.420020,1013.420020,0.8,0.8,312.146667,312.146667
2018-06-04 00:00:00+00:00,0.0,2011.500000,1011.500000,1.8,0.9,531.528428,171.528428
2018-06-05 00:00:00+00:00,0.0,2007.450345,1007.450345,0.9,0.9,300.256667,300.256667
2018-06-06 00:00:00+00:00,0.0,2015.010331,1015.010331,0.8,0.8,136.380000,136.380000


Prepared dataset shape: (131617, 16)


## 7. Define the historical forecasting inputs

The next notebooks will build F0–F4 feature sets. Their starting point, **F0**, consists of recent observations of PV power and weather.

The nine historical sources below are measured variables, so a model may only use them at the **forecast issue time or earlier**. Later notebooks will construct historical lags and a previous-day PV lag; they will not use observed weather from the forecast-valid time.

Deterministic Gregorian, solar, lunar, and Panchang features will be built separately using time and location. Those features can describe the future forecast-valid time without looking at future measured weather.

In [7]:
historical_sources = {
    "PV power": "P_Solar[kW]",
    "Global horizontal irradiance": "GHI[kW1m2]",
    "Plane-of-array irradiance": "POA Irr[kW1m2]",
    "Air temperature": "TEMPERATURE[degC]",
    "Humidity fraction": "Humidity_clean",
    "Atmospheric pressure": "Pressure_clean[mbar]",
    "Wind speed": "WIND_SPEED[m1s]",
    "Wind direction sine": "wind_direction_sin",
    "Wind direction cosine": "wind_direction_cos",
}

assert len(historical_sources) == 9
assert set(historical_sources.values()).issubset(cleaned.columns)
assert np.isfinite(cleaned[list(historical_sources.values())].to_numpy(dtype=float)).all()

source_table = pd.DataFrame(
    {"source_column": historical_sources, "available_to_model": "At or before forecast issue time"}
)
display(source_table)

,source_column,available_to_model
PV power,P_Solar[kW],At or before forecast issue time
Global horizontal irradiance,GHI[kW1m2],At or before forecast issue time
Plane-of-array irradiance,POA Irr[kW1m2],At or before forecast issue time
Air temperature,TEMPERATURE[degC],At or before forecast issue time
Humidity fraction,Humidity_clean,At or before forecast issue time
Atmospheric pressure,Pressure_clean[mbar],At or before forecast issue time
Wind speed,WIND_SPEED[m1s],At or before forecast issue time
Wind direction sine,wind_direction_sin,At or before forecast issue time
Wind direction cosine,wind_direction_cos,At or before forecast issue time


## 8. Final quality checks

Before moving on, we verify that the cleaned variables have no missing values, the time index has not changed, and the original measurements have been preserved.

These checks confirm **data preparation only**. They do not establish forecasting accuracy or any benefit from Panchang features.

In [8]:
added = [
    "Pressure_clean[mbar]", "Humidity_clean", "Wind_dir_clean[deg]",
    "wind_direction_sin", "wind_direction_cos",
]
verification = pd.Series(
    {
        "observations": len(cleaned),
        "original_measurement_columns": len(raw.columns),
        "new_cleaned_columns": len(added),
        "pressure_rows_corrected": int(pressure_issue.sum()),
        "humidity_rows_replaced": int(humidity_issue.sum()),
        "wind_directions_wrapped": int(wind_issue.sum()),
        "missing_values_in_prepared_inputs": int(cleaned[list(historical_sources.values())].isna().sum().sum()),
        "original_columns_preserved": all(cleaned[c].equals(raw[c]) for c in raw.columns),
        "timeline_preserved": cleaned.index.equals(raw.index),
    },
    name="Data preparation checks",
)
display(verification.to_frame("value"))

assert all(np.isfinite(cleaned[col].to_numpy(dtype=float)).all() for col in added)
assert verification["original_columns_preserved"]
assert verification["timeline_preserved"]
assert verification["missing_values_in_prepared_inputs"] == 0
assert cleaned["P_Solar[kW]"].equals(raw["P_Solar[kW]"])

print("NOTEBOOK 02: DATA PREPARATION CHECKS PASSED")

,value
observations,131617
original_measurement_columns,11
new_cleaned_columns,5
pressure_rows_corrected,456
humidity_rows_replaced,164
wind_directions_wrapped,103
missing_values_in_prepared_inputs,0
original_columns_preserved,True
timeline_preserved,True


NOTEBOOK 02: DATA PREPARATION CHECKS PASSED


## What we learned

- The 5-minute SOLETE measurements can be prepared without changing the original source columns.
- The known pressure artifact is corrected only where it occurs.
- Invalid humidity values are replaced using **past** observations rather than future values.
- Wind direction is represented continuously with sine and cosine.
- The nine observational inputs needed for F0 are now defined.

**Next:** Notebook 03 will construct the deterministic calendar, solar, and lunar feature groups. We will build the actual forecasting samples and historical lags in the forecasting-dataset notebook, so the distinction between **issue time** and **forecast-valid time** stays clear.